# 02 — Initial Exploratory Data Analysis

This notebook performs exploratory data analysis on the cleaned and transformed World Bank indicator datasets.

**Inputs:** Cleaned CSVs from `01_ingestion_and_imputation/`  
**Outputs:** Distribution plots, missing-value heatmaps, correlation heatmaps saved to this folder

**Datasets:**
- Economic Indicators
- Environmental Indicators
- Public Debt Indicators
- Social Indicators
- Statistical Indicators

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Load Cleaned Datasets

In [ ]:
# Paths point to cleaned outputs from 01_ingestion_and_imputation
DATA_DIR = "../01_ingestion_and_imputation/"

file_paths = {
    "Economic_Indicators":    DATA_DIR + "Economic_Indicators_transformed.csv",
    "Environmental_Indicators": DATA_DIR + "Environmental_Indicators_transformed.csv",
    "Public_Debt_Indicators":  DATA_DIR + "Public_Debt_Indicators_transformed.csv",
    "Social_Indicators":       DATA_DIR + "Social_Indicators_transformed.csv",
    "Statistical_Indicators":  DATA_DIR + "Statistical_Indicators_transformed.csv",
}

dfs = {}
for name, path in file_paths.items():
    if os.path.exists(path):
        dfs[name] = pd.read_csv(path)
        print(f"Loaded {name}: {dfs[name].shape}")
    else:
        print(f"Not found: {path}")

## 2. Missing Value Analysis

In [ ]:
# Missing value percentage summary
missing_pct = {
    name: round((df.isnull().sum().sum() / df.size) * 100, 2)
    for name, df in dfs.items()
}
missing_df = pd.DataFrame(list(missing_pct.items()), columns=["Dataset", "Missing %"])
print(missing_df.to_string(index=False))

In [ ]:
# Missing value heatmaps per dataset
for name, df in dfs.items():
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False, ax=ax)
    ax.set_title(f'Missing Values — {name}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'missing_values_{name}.png', bbox_inches='tight')
    plt.show()

## 3. Descriptive Statistics

In [ ]:
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"Descriptive Statistics — {name}")
    print('='*60)
    display(df.describe())

## 4. Value Distributions (Histograms)

In [ ]:
for name, df in dfs.items():
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) == 0:
        continue
    df[numeric_cols].hist(figsize=(14, 8), bins=30, color='steelblue', edgecolor='white')
    plt.suptitle(f'Value Distribution — {name}', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(f'distribution_{name}.png', bbox_inches='tight')
    plt.show()

## 5. Outlier Detection (Boxplots)

In [ ]:
for name, df in dfs.items():
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) == 0:
        continue
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=df[numeric_cols], ax=ax, palette='Set2')
    ax.set_title(f'Boxplot — {name}', fontsize=13)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'boxplot_{name}.png', bbox_inches='tight')
    plt.show()

## 6. Correlation Heatmaps

In [ ]:
for name, df in dfs.items():
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        continue
    corr = numeric_df.corr()
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(
        corr, annot=True, fmt='.2f', cmap='coolwarm',
        linewidths=0.5, ax=ax, annot_kws={'size': 8}
    )
    ax.set_title(f'Correlation Heatmap — {name}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'correlation_{name}.png', bbox_inches='tight')
    plt.show()

## 7. Regional and Indicator Coverage

The PNGs `regional_coverage.png` and `regional_indicator_coverage.png` in this folder were generated separately using the `pycountry` library to map `country_code` → region, then aggregating indicator counts per region.

In [ ]:
# Display the pre-generated regional coverage plots
from IPython.display import Image, display as ipy_display

for img_path in ['regional_coverage.png', 'regional_indicator_coverage.png', 'economic_values_distribution.png']:
    if os.path.exists(img_path):
        print(f"\n{img_path}")
        ipy_display(Image(img_path))

## Summary

| Finding | Detail |
|---------|--------|
| Highest missingness | Public Debt Indicators (~quarterly data with sparse coverage) |
| Strongest correlations | GDP-related economic indicators cluster together |
| Outlier risk | Financial indicators show heavy right skew |
| Country coverage | 217 countries across 7 World Bank regions |

All plots saved as PNGs in this directory for use in the final report.